# Conclusions & Submission

## Findings

| Cluster | Rows | Nature |
|---|---|---|
| `bank_id == 777` | 635 | synthetic block (constant 5s latency, 1 day, 100% fail) |
| psp_gamma latency | 5,433 | real provider outage (12-17 May, gradual degrade/recover) |
| psp_beta over-refund | 2,691 | synthetic/corrupted (+10 flat, 5-9 Aug) |
| fail + refund | 1,299 | synthetic (94% exact 0.5x ratio, all year) |

Ruled out after investigation: `ip_country`/`bin_country` mismatch, per-category amount "outliers" (a legitimate price ladder), `psp_alpha`'s systemic fail rate — all explainable, none concentrated enough to be a labeling signal.

IsolationForest (`03_ml_scoring.ipynb`) independently confirms 3 of the 4 clusters and structurally cannot see the 4th (a collective, not point, anomaly) — no 5th cluster found in the model's residual top 1%.

**Labeling decision:** the psp_gamma incident is a real operational event, not a data anomaly — excluded from `is_anomaly`. The other 3 are flagged.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                      'axes.spines.right': False, 'figure.facecolor': 'white'})

DATA_PATH = '/home/veronika/Anomaly_Hunter_Solidgate/hackathon_int20h_dataset_test.csv'

df = pd.read_csv(DATA_PATH)
df['created_at'] = pd.to_datetime(df['created_at'])
df['processed_at'] = pd.to_datetime(df['processed_at'])

In [2]:
is_anomaly = (
    (df['bank_id'] == 777) |
    (df['refunded_amount'] > df['amount']) |
    (df['has_refund'] & (df['status'] == 'fail'))
)

submission = pd.DataFrame({'order_id': df['order_id'], 'is_anomaly': is_anomaly.astype(int)})
submission.to_csv('/home/veronika/Anomaly_Hunter_Solidgate/submission.csv', index=False)

print(f"is_anomaly rate: {is_anomaly.mean():.3%}  ({is_anomaly.sum():,} of {len(df):,} rows)")
print(submission.head())
print("\nSaved: submission.csv")

is_anomaly rate: 0.456%  (4,559 of 1,000,000 rows)
   order_id  is_anomaly
0         1           0
1         2           0
2         3           0
3         4           0
4         5           0

Saved: submission.csv
